In [1]:
import math
import pandas as pd
from chemplus import mol_df
from chemplus.vina import exprank
pd.set_option("display.max_colwidth", 100)

#J/(K*mol) / kcal/J -> kcal/(K*mol)
kcal_to_jole = 4184
# Universal gas constant
R = 8.31432
# Temperature, K
T = 298.5

# Set parameters for ECR calculation

In [2]:
docked_sdf_list = ["docking/enamine_docked.sdf.gz", "Ensitrelvir/ensitrelvir_5-confs_docked.sdf.gz"]
dockscores_list = ["docking/dockscore.csv", "Ensitrelvir/dockscore.csv"]

# Read data

In [3]:
df = pd.concat([pd.read_csv(dockscore_csv_path) for dockscore_csv_path in dockscores_list])
df["Short name"] = df["Name"].apply(lambda x: x.split("_conf-")[0])
df = df.loc[:, ["Name", "Short name", "E_vina, kcal/mol", "pKd_vina", "pKd_on2", "pKd_pl", "pKd_pn2"]]

# Calculate ECR

sigmas is from 1% to 5% of samples (N=1000 => sigmas=10)

In [ ]:
ecr_sf_columns = ["pKd_on2", "pKd_pl"]
sf_sigmas = [7000, 7000]
out_ecr_csv = "ecr_ranking.csv"

#

e_cols = []
for sf in ecr_sf_columns:
    e_name = f"E_{sf.split('_')[1]}, kcal/mol"
    e_cols.append(e_name)
    df[e_name] = df[sf].apply(lambda x: -x*R*T*math.log(10)/kcal_to_jole)

df = exprank.get_exprank_df(df, sf_columns_list=ecr_sf_columns, sf_sigmas_list=sf_sigmas)
df.to_csv(out_ecr_csv, index=False)

# Check data

In [11]:
columns_to_show = ["Name", "E_vina, kcal/mol", "E_on2, kcal/mol", "E_pl, kcal/mol",
                   "pKd_on2 rank", "pKd_pl rank", "ECR"]
rounding = {"E_vina, kcal/mol" : 1, "E_on2, kcal/mol" : 1, "E_pl, kcal/mol" : 1, "ECR" : 6}
df[~df["Short name"].duplicated()][columns_to_show].round(rounding).head(25)

,Name,"E_vina, kcal/mol","E_on2, kcal/mol","E_pl, kcal/mol",pKd_on2 rank,pKd_pl rank,ECR
0,Z5977622777,-8.4,-10.5,-13.8,103.0,42.0,0.000283
1,Z1489210026,-7.4,-10.3,-14.1,371.0,25.0,0.000278
2,Z1668961048,-8.2,-10.9,-12.6,3.0,438.0,0.000277
3,Z220018612,-7.9,-10.4,-12.8,227.0,308.0,0.000275
4,Z5575926508_frag-1,-8.1,-10.5,-12.7,139.0,411.0,0.000275
5,Z80597289,-8.6,-10.3,-12.9,515.0,246.0,0.000271
6,Z8416259464,-8.7,-10.4,-12.5,174.0,603.0,0.000270
7,Z319868382,-8.3,-10.4,-12.5,254.0,546.0,0.000270
8,Z3278678612,-7.9,-10.2,-12.9,780.0,261.0,0.000265
9,Z109938490,-7.3,-10.2,-12.9,824.0,241.0,0.000265


# Save top

In [23]:
from IPython.display import HTML
top_num = 200
out_top_csv = "top200/ligs.csv"
out_top_sdf = "top200/ligs.sdf"

#####

df_top = df[~df.duplicated("Short name") & ~df["Name"].str.contains("Ensitrelvir")].head(top_num).copy(deep=True)
df_sdf_top = pd.concat([mol_df.df_from_sdf(docked_sdf_list[1], mol_names=["Ensitrelvir_conf-1"]), 
                        mol_df.df_from_sdf(docked_sdf_list[0], mol_names=list(df_top["Name"].values))])
mol_df.df_to_sdf(df_sdf_top, out_top_sdf)
df_top = pd.concat([df[df["Name"].str.contains("Ensitrelvir")].head(1), df_top])
df_top = df_top[columns_to_show]
df_top = df_top.round(rounding)
df_top.to_csv(out_top_csv, index=False)
HTML(df_top.to_html(index=False))

Name,"E_vina, kcal/mol","E_on2, kcal/mol","E_pl, kcal/mol",pKd_on2 rank,pKd_pl rank,ECR
Ensitrelvir_conf-1,-8.5,-10.4,-11.5,223.0,4813.0,0.000210
Z5977622777,-8.4,-10.5,-13.8,103.0,42.0,0.000283
Z1489210026,-7.4,-10.3,-14.1,371.0,25.0,0.000278
Z1668961048,-8.2,-10.9,-12.6,3.0,438.0,0.000277
Z220018612,-7.9,-10.4,-12.8,227.0,308.0,0.000275
Z5575926508_frag-1,-8.1,-10.5,-12.7,139.0,411.0,0.000275
Z80597289,-8.6,-10.3,-12.9,515.0,246.0,0.000271
Z8416259464,-8.7,-10.4,-12.5,174.0,603.0,0.000270
Z319868382,-8.3,-10.4,-12.5,254.0,546.0,0.000270
Z3278678612,-7.9,-10.2,-12.9,780.0,261.0,0.000265
